# Zero123++ v1.2 з одного фото

Ноутбук готує фото та створює одну PNG-сітку **640 × 960** із шістьма
ракурсами **320 × 320**. Можна завантажити одне фото або прогнати десять
фото з набору Студента 2. Для кожного входу зберігаються підготовлене фото,
сітка й запис у `run.json`.

1. Відкрийте цей `.ipynb` у Colab через **File → Upload notebook**.
2. У **Runtime → Change runtime type** виберіть **T4 GPU** в **Hardware accelerator**
   та **2026.07** у **Runtime Version**. Ця версія Colab містить Python **3.12.13**.
   Під'єднайтеся до вибраного середовища. Для залежностей і ваг потрібно
   приблизно 15 ГБ вільного місця. [Версії середовищ Colab](https://research.google.com/colaboratory/runtime-version-faq.html).
3. Виберіть `MODE = "upload"` або `MODE = "batch"` і виконайте комірки зверху вниз.
4. Збережіть ZIP з результатами до завершення сесії. Дані `/content` тимчасові.

Ваги v1.2 завантажуються з Hugging Face за закріпленим SHA (приблизно 5,58 ГБ).
Код моделі Apache 2.0, ваги CC BY-NC 4.0 для некомерційного використання.
[Офіційний Zero123++](https://github.com/SUDO-AI-3D/zero123plus),
[ваги v1.2](https://huggingface.co/sudo-ai/zero123plus-v1.2),
[код пайплайна](https://huggingface.co/sudo-ai/zero123plus-pipeline).


In [ ]:
import hashlib
import json
import os
from pathlib import Path
import shutil
import subprocess
import sys
from urllib.request import urlopen
from uuid import uuid4
import venv

MODE = "upload"  # "upload" або "batch"
SEED = 42
STEPS = 28
VAE_TILING = False  # Увімкніть, якщо бракує VRAM під час декодування.

if MODE not in {"upload", "batch"}:
    raise ValueError('MODE має бути "upload" або "batch".')
if sys.version_info[:2] != (3, 12):
    raise RuntimeError(
        f"Поточний Python {sys.version.split()[0]}. Потрібен Python 3.12. "
        "У Runtime → Change runtime type виберіть T4 GPU та Runtime Version → 2026.07, "
        "перепідключіться й запустіть ноутбук з початку."
    )

try:
    subprocess.run(["nvidia-smi", "--query-gpu=name,memory.free", "--format=csv"], check=True)
except (OSError, subprocess.CalledProcessError) as exc:
    raise RuntimeError("Виберіть GPU в Runtime → Change runtime type і перепідключіться.") from exc

WORK_DIR = Path("/content/photo3d_zero123")
WORK_DIR.mkdir(parents=True, exist_ok=True)
ENV_DIR = WORK_DIR / "runtime"
ENV_PYTHON = ENV_DIR / "bin" / "python"
RUN_ENV = dict(os.environ, HF_HOME=str(WORK_DIR / "hf_cache"), HF_HUB_DOWNLOAD_TIMEOUT="120",
               U2NET_HOME=str(WORK_DIR / "rembg_cache"))


## Ізольоване середовище

У комірках нижче записано закріплені залежності та повний код запуску.
Ноутбук запускається до злиття гілки в `main`. Пакети встановлюються у
власне середовище, тому версії NumPy в Colab та в основному проєкті
можуть залишатися різними. Перший запуск завантажує CUDA-пакети й ваги.


In [ ]:
%%writefile /content/photo3d_zero123/zero123_requirements.in
--extra-index-url https://download.pytorch.org/whl/cu121
torch==2.5.1+cu121
torchvision==0.20.1+cu121
diffusers==0.27.2
transformers==4.39.3
huggingface-hub==0.25.2
accelerate==0.27.2
safetensors==0.4.5
numpy==1.26.4
pillow==10.4.0
rembg[cpu]==2.0.61
onnxruntime==1.20.1
opencv-python-headless==4.10.0.84
scipy==1.14.1
scikit-image==0.24.0
pymatting==1.1.12
numba==0.60.0
llvmlite==0.43.0
tokenizers==0.15.2


In [ ]:
%%writefile /content/photo3d_zero123/run_zero123.py
"""Generate the original six-view Zero123++ v1.2 grid on a CUDA GPU."""

import argparse
import hashlib
import importlib.metadata
import json
import os
from pathlib import Path
import re
import subprocess
import sys
import time
from urllib.request import urlopen


MODEL_ID = "sudo-ai/zero123plus-v1.2"
MODEL_REVISION = "2da07e89919e1a130c9b5add1584c70c7aa065fd"
PIPELINE_ID = "sudo-ai/zero123plus-pipeline"
PIPELINE_REVISION = "983e66d28a3637ddd8e3e2fd8165cdff32230872"
DATASET_REVISION = "e2f81456de75b15aec6d4839bb1ecb9c44c73177"
DATASET_ROOT = f"https://raw.githubusercontent.com/Lenuar23/3d-model_generator/{DATASET_REVISION}"
MASK_MODEL = "u2netp"
INPUT_SIZE = 320
GRID_SIZE = (640, 960)
GRAY = (127, 127, 127)


def sha256(path):
    with path.open("rb") as stream:
        return hashlib.file_digest(stream, "sha256").hexdigest()


def load_batch(photos_dir):
    with urlopen(f"{DATASET_ROOT}/docs/test_photos_manifest.json", timeout=30) as response:
        photos = json.load(response)["photos"]
    filenames = [photo["filename"] for photo in photos]
    if len(filenames) != 10 or len(set(filenames)) != 10:
        raise ValueError("Expected the ten distinct photographs in the pinned manifest")
    inputs = []
    for photo in photos:
        name = photo["filename"]
        if not re.fullmatch(r"[a-zA-Z0-9_-]+\.jpg", name):
            raise ValueError(f"Unexpected manifest filename: {name}")
        path = photos_dir / name
        if not path.is_file() or sha256(path) != photo["sha256"]:
            raise ValueError(f"Missing or changed test photograph: {path}")
        inputs.append((path, photo["category"], photo["sha256"]))
    return inputs


def condition_photo(path, segment):
    from PIL import Image, ImageOps

    with Image.open(path) as original:
        photo = ImageOps.exif_transpose(original).convert("RGBA")
    photo.thumbnail((1024, 1024), Image.Resampling.LANCZOS)
    alpha = photo.getchannel("A")
    if alpha.getextrema()[0] < 255:
        foreground = photo
        method = "source_alpha"
    else:
        foreground = segment(photo.convert("RGB")).convert("RGBA")
        method = "rembg_u2netp"
    alpha = foreground.getchannel("A")
    bounds = alpha.point(lambda value: 255 if value >= 16 else 0).getbbox()
    if bounds is None:
        if method == "source_alpha":
            raise ValueError("Input photo has no visible pixels")
        # Preserve a useful conditioning input when glass defeats segmentation.
        foreground = photo
        method = "original_rgb_empty_mask"
    else:
        foreground = foreground.crop(bounds)
    foreground = ImageOps.contain(foreground, (256, 256), Image.Resampling.LANCZOS)
    canvas = Image.new("RGB", (INPUT_SIZE, INPUT_SIZE), GRAY)
    offset = ((INPUT_SIZE - foreground.width) // 2, (INPUT_SIZE - foreground.height) // 2)
    canvas.paste(foreground, offset, foreground.getchannel("A"))
    return canvas, method


def write_json(path, data):
    temporary = path.with_suffix(".tmp")
    temporary.write_text(json.dumps(data, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
    temporary.replace(path)


def prepare_photos(output_dir):
    # Process rembg in a separate interpreter so its CPU memory is released before CUDA load.
    from rembg import new_session, remove

    report_path = output_dir / "run.json"
    report = json.loads(report_path.read_text(encoding="utf-8"))
    session = None

    def segment(image):
        nonlocal session
        if session is None:
            session = new_session(MASK_MODEL, providers=["CPUExecutionProvider"])
        return remove(image, session=session)

    for index, record in enumerate(report["photos"], start=1):
        try:
            photo, method = condition_photo(Path(record["input"]), segment)
            filename = f"{index:02d}-{Path(record['input']).stem}-condition.png"
            photo.save(output_dir / filename)
            record.update(status="prepared", condition=filename, preprocessing=method)
            print(f"prepared {index}/{len(report['photos'])}: {record['input']}", flush=True)
        except Exception as exc:
            record.update(status="failed", stage="preprocessing", error=f"{type(exc).__name__}: {exc}")
            print(f"preprocessing failed: {record['input']}: {exc}", file=sys.stderr, flush=True)
        write_json(report_path, report)


def load_pipeline(torch, vae_tiling):
    from diffusers import DiffusionPipeline, EulerAncestralDiscreteScheduler
    from huggingface_hub import hf_hub_download

    custom_code = hf_hub_download(PIPELINE_ID, "pipeline.py", revision=PIPELINE_REVISION)
    pipe = DiffusionPipeline.from_pretrained(
        MODEL_ID, revision=MODEL_REVISION, custom_pipeline=str(Path(custom_code).parent),
        torch_dtype=torch.float16, use_safetensors=True, low_cpu_mem_usage=True,
    )
    pipe.scheduler = EulerAncestralDiscreteScheduler.from_config(
        pipe.scheduler.config, timestep_spacing="trailing",
    )
    if vae_tiling:
        pipe.enable_vae_tiling()
    return pipe.to("cuda")


def generate_grid(pipe, torch, photo, seed, steps):
    # Upstream condition latents and reference UNet noise also use global Torch RNG.
    torch.manual_seed(seed)
    generator = torch.Generator(device="cuda").manual_seed(seed)
    with torch.inference_mode():
        grid = pipe(photo, num_inference_steps=steps, guidance_scale=4.0, generator=generator).images[0]
    if grid.size != GRID_SIZE:
        raise ValueError(f"Expected grid {GRID_SIZE}, got {grid.size}")
    grid = grid.convert("RGB")
    if all(low == high for low, high in grid.getextrema()):
        raise ValueError("Zero123++ returned a uniform grid")
    return grid


def run(inputs, output_dir, seed, steps, vae_tiling, batch):
    from PIL import Image

    output_dir.mkdir(parents=True, exist_ok=False)
    report_path = output_dir / "run.json"
    report = dict(
        status="started", model=MODEL_ID, model_revision=MODEL_REVISION,
        pipeline=PIPELINE_ID, pipeline_revision=PIPELINE_REVISION, mask_model=MASK_MODEL,
        dataset_revision=DATASET_REVISION if batch else None, seed=seed, inference_steps=steps,
        guidance_scale=4.0, vae_tiling=vae_tiling, grid_size=list(GRID_SIZE),
        photos=[dict(input=str(path.resolve()), category=category, input_sha256=digest,
                     status="pending") for path, category, digest in inputs],
    )
    write_json(report_path, report)
    try:
        child_env = dict(os.environ, NUMBA_DISABLE_JIT="1", OMP_NUM_THREADS="2")
        try:
            subprocess.run([sys.executable, str(Path(__file__).resolve()), "--prepare-dir", str(output_dir)],
                           env=child_env, check=True)
        finally:
            # A failed subprocess may already have recorded preparation progress.
            report = json.loads(report_path.read_text(encoding="utf-8"))
        if not any(record["status"] == "prepared" for record in report["photos"]):
            raise RuntimeError("None of the input photographs could be prepared")
        import torch

        if not torch.cuda.is_available():
            raise RuntimeError("CUDA is unavailable. Select a Colab GPU runtime or install CUDA PyTorch.")
        free, total = torch.cuda.mem_get_info()
        report["environment"] = dict(
            python=sys.version.split()[0], gpu=torch.cuda.get_device_name(0), cuda=torch.version.cuda,
            vram_free_bytes_at_start=free, vram_total_bytes=total,
            packages={name: importlib.metadata.version(name) for name in (
                "torch", "torchvision", "diffusers", "transformers", "huggingface-hub", "accelerate",
                "numpy", "Pillow", "rembg", "onnxruntime", "safetensors",
            )},
        )
        write_json(report_path, report)
        print(f"Loading {MODEL_ID} on {report['environment']['gpu']} (FP16)", flush=True)
        pipe = load_pipeline(torch, vae_tiling)
        for index, record in enumerate(report["photos"], start=1):
            if record["status"] != "prepared":
                continue
            print(f"generating {index}/{len(report['photos'])}: {record['input']}", flush=True)
            try:
                torch.cuda.reset_peak_memory_stats()
                start = time.perf_counter()
                with Image.open(output_dir / record["condition"]) as condition:
                    grid = generate_grid(pipe, torch, condition, seed, steps)
                torch.cuda.synchronize()
                name = f"{index:02d}-{Path(record['input']).stem}-grid.png"
                target = output_dir / name
                grid.save(target)
                record.update(status="succeeded", grid=name, grid_sha256=sha256(target),
                              seconds=round(time.perf_counter() - start, 3),
                              peak_vram_allocated_bytes=torch.cuda.max_memory_allocated())
            except Exception as exc:
                record.update(status="failed", stage="inference", error=f"{type(exc).__name__}: {exc}")
                print(f"inference failed: {record['input']}: {exc}", file=sys.stderr, flush=True)
                torch.cuda.empty_cache()
            write_json(report_path, report)
    except Exception as exc:
        report["fatal_error"] = f"{type(exc).__name__}: {exc}"
        report["status"] = "failed"
        write_json(report_path, report)
        raise
    succeeded = sum(record["status"] == "succeeded" for record in report["photos"])
    report.update(status="succeeded" if succeeded == len(inputs) else "failed", succeeded=succeeded)
    write_json(report_path, report)
    print(f"Saved {succeeded}/{len(inputs)} grids. Report: {report_path}", flush=True)
    return succeeded == len(inputs)


def main():
    parser = argparse.ArgumentParser(description=__doc__)
    mode = parser.add_mutually_exclusive_group(required=True)
    mode.add_argument("--photo", type=Path)
    mode.add_argument("--batch", action="store_true")
    mode.add_argument("--prepare-dir", type=Path, help=argparse.SUPPRESS)
    parser.add_argument("--photos-dir", type=Path, default=Path("data/test_photos"))
    parser.add_argument("--output-dir", type=Path)
    parser.add_argument("--seed", type=int, default=42)
    parser.add_argument("--steps", type=int, default=28)
    parser.add_argument("--vae-tiling", action="store_true")
    args = parser.parse_args()
    if args.prepare_dir:
        prepare_photos(args.prepare_dir)
        return 0
    if args.output_dir is None:
        parser.error("--output-dir is required")
    if not 0 <= args.seed < 2**63 or not 1 <= args.steps <= 1000:
        parser.error("Choose a seed in [0, 2**63) and steps in [1, 1000]")
    inputs = (load_batch(args.photos_dir) if args.batch
              else [(args.photo, None, sha256(args.photo))])
    return 0 if run(inputs, args.output_dir.resolve(), args.seed, args.steps, args.vae_tiling, args.batch) else 1


if __name__ == "__main__":
    sys.exit(main())


In [ ]:
if not ENV_PYTHON.is_file():
    if shutil.disk_usage(WORK_DIR).free < 15 * 1024**3:
        raise RuntimeError("Звільніть приблизно 15 ГБ у runtime для CUDA-пакетів і ваг.")
    venv.EnvBuilder(with_pip=False).create(ENV_DIR)

pip = [sys.executable, "-m", "pip", "--python", str(ENV_PYTHON)]
subprocess.run(pip + ["install", "--no-cache-dir", "-r", str(WORK_DIR / "zero123_requirements.in")], check=True)
subprocess.run(pip + ["check"], check=True)
cuda_check = (
    "import torch; "
    "assert torch.cuda.is_available(), 'CUDA недоступна в ізольованому середовищі'; "
    "x = torch.tensor([1., 2., 3.], device='cuda'); "
    "assert (x*x).sum().item() == 14; "
    "print(torch.cuda.get_device_name(0), torch.__version__)"
)
subprocess.run([str(ENV_PYTHON), "-c", cuda_check], env=RUN_ENV, check=True)


## Вибір фото

Для одного фото відкриється діалог завантаження. Для набору ноутбук
отримає маніфест і завантажувач з конкретного коміту завдання 3 та перевірить
SHA-256 усіх десяти фото. Код працює з одним входом за раз.

Фото повертається за EXIF. Якщо в ньому є прозорість, вона зберігається під
час підготовки. Для непрозорого фото `rembg` застосовує `u2netp` на CPU.
Об'єкт центрується на сірому квадраті 320 × 320 без зміни пропорцій. Якщо
маска порожня, береться початкове фото й це позначається в `run.json`.


In [ ]:
if MODE == "upload":
    from google.colab import files

    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("Завантажте рівно одне фото.")
    name, payload = next(iter(uploaded.items()))
    upload_dir = WORK_DIR / "uploads" / uuid4().hex
    upload_dir.mkdir(parents=True)
    photo_path = upload_dir / Path(name).name
    photo_path.write_bytes(payload)
    input_args = ["--photo", str(photo_path)]
else:
    dataset_revision = "e2f81456de75b15aec6d4839bb1ecb9c44c73177"
    dataset_base = f"https://raw.githubusercontent.com/Lenuar23/3d-model_generator/{dataset_revision}"
    assets = {
        "docs/test_photos_manifest.json": "7971c6a504f11b4ca8276bc0f110662f689cf67c12e119efe2432311f601821e",
        "scripts/download_test_photos.py": "b545d2eb407b1e39f6d2ad26359966e8d4238aa3b560834920b8f9c2abff1085",
    }
    for relative_path, expected_hash in assets.items():
        target = WORK_DIR / relative_path
        if target.is_file():
            payload = target.read_bytes()
        else:
            with urlopen(f"{dataset_base}/{relative_path}", timeout=30) as response:
                payload = response.read()
        if hashlib.sha256(payload).hexdigest() != expected_hash:
            raise RuntimeError(f"SHA-256 не збігається для {relative_path}.")
        if not target.exists():
            target.parent.mkdir(parents=True, exist_ok=True)
            target.write_bytes(payload)
    photos_dir = WORK_DIR / "data" / "test_photos"
    subprocess.run([str(ENV_PYTHON), str(WORK_DIR / "scripts/download_test_photos.py"),
                    "--output-dir", str(photos_dir)], check=True)
    input_args = ["--batch", "--photos-dir", str(photos_dir)]


## Генерація та результати

Один запуск використовує фіксований seed для кожного фото. Результати
зберігаються в новій папці. У `run.json` є версії пакетів, GPU, ревізії,
SHA-256 входів і сіток, тривалість, пікова виділена VRAM та помилки.
У режимі `batch` успіх означає, що збережено всі десять валідних сіток.


In [ ]:
RUN_DIR = WORK_DIR / "runs" / uuid4().hex
command = [str(ENV_PYTHON), "-u", str(WORK_DIR / "run_zero123.py"), *input_args,
           "--output-dir", str(RUN_DIR), "--seed", str(SEED), "--steps", str(STEPS)]
if VAE_TILING:
    command.append("--vae-tiling")
completed = subprocess.run(command, env=RUN_ENV)
report_path = RUN_DIR / "run.json"
if report_path.is_file():
    report = json.loads(report_path.read_text(encoding="utf-8"))
    print(json.dumps(report, ensure_ascii=False, indent=2))
if completed.returncode != 0:
    raise RuntimeError(f"Генерація завершилася з помилкою. Діагностика: {report_path}")

from IPython.display import Image, display
display(Image(filename=str(RUN_DIR / report["photos"][0]["grid"])))


## Зберегти ZIP

Комірка архівує наявні файли запуску, включно з `run.json`, і відкриває
завантаження ZIP. Після збою можна запустити її окремо, щоб зберегти діагностику.


In [ ]:
from google.colab import files

if not RUN_DIR.is_dir():
    raise RuntimeError("Папка запуску відсутня. Спочатку виконайте генерацію.")
archive = shutil.make_archive(str(RUN_DIR), "zip", root_dir=RUN_DIR)
files.download(archive)
